In [29]:
import  pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import ExtraTreesClassifier

In [11]:
df = pd.read_csv("../data/cleaned/cleaned_credit_default.csv")

y = df["SeriousDlqin2yrs"]
X = df.drop(columns=["SeriousDlqin2yrs"])

categorical_cols = ["AgeGroup"]
numeric_cols = X.columns.difference(categorical_cols).tolist()

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_cat = encoder.fit_transform(X[categorical_cols])
X_num = X[numeric_cols].to_numpy()

X_final = np.hstack([X_num, X_cat])

X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (119212, 24)
X_test: (29804, 24)


## 1. Decision Tree

In [4]:
dt = DecisionTreeClassifier(
    random_state=42
)

param_grid = {
    "max_depth": [5 , 7 , 10],
    "min_samples_split": [2 , 10 , 50],
    "min_samples_leaf": [1 , 5 ,20],
    "class_weight": [
        None,
        {0 : 1 , 1 : 2},
        {0 : 1 , 1 : 3},
        {0 : 1 , 1 : 5},
    ],
    "ccp_alpha": [0, 0.00005, 0.0001]
}

grid_search = GridSearchCV(
    estimator = dt,
    param_grid = param_grid,
    scoring = "f1",
    cv = 5,
    n_jobs = -1,
    verbose = 1
)
grid_search.fit(X_train, y_train)
print("Best F1:", grid_search.best_score_)
print("Best Parameters:", grid_search.best_params_)

Fitting 5 folds for each of 324 candidates, totalling 1620 fits
Best F1: 0.4326349389977754
Best Parameters: {'ccp_alpha': 0.0001, 'class_weight': {0: 1, 1: 3}, 'max_depth': 7, 'min_samples_leaf': 20, 'min_samples_split': 2}


In [10]:
best_dt = grid_search.best_estimator_

y_pred = best_dt.predict(X_test)
y_prob = best_dt.predict_proba(X_test)[ : , 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1       :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

Accuracy : 0.9207824453093545
Precision: 0.40825035561877665
Recall   : 0.43617021276595747
F1       : 0.42174871418074944
ROC-AUC  : 0.8485663244892914


### GridSearchCV — Decision Tree Results

| Metric | Score |
|---|---:|
| **Accuracy** | **0.920782** |
| **Precision** | **0.408250** |
| **Recall** | **0.436170** |
| **F1 Score** | **0.421749** |
| **ROC-AUC** | **0.848566** |

## 2. Random Forest

In [15]:
rf = RandomForestClassifier(
    random_state=42
)

rf_param_grid = {
    "n_estimators" : [100 , 200],
    "max_depth" : [7 , 10 , None],
    "min_samples_leaf" : [1 , 5 ,20],
    "class_weight": [
        None,
        {0: 1, 1: 2},
        {0: 1, 1: 3}
    ]
}

rf_grid = GridSearchCV(
    estimator = rf,
    param_grid = rf_param_grid,
    scoring = 'f1',
    cv = 5,
    n_jobs = -1,
    verbose = 1
)

rf_grid.fit(X_train , y_train)

print("Best F1:", rf_grid.best_score_)
print("Best Parameters:", rf_grid.best_params_)

Fitting 5 folds for each of 54 candidates, totalling 270 fits


C:\Users\Rupali Waghmare\miniconda3\envs\AI_ML\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best F1: 0.43392954854499877
Best Parameters: {'class_weight': {0: 1, 1: 3}, 'max_depth': 10, 'min_samples_leaf': 20, 'n_estimators': 200}


In [16]:
best_rf = rf_grid.best_estimator_

y_pred_rf = best_rf.predict(X_test)
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall   :", recall_score(y_test, y_pred_rf))
print("F1       :", f1_score(y_test, y_pred_rf))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_rf))

Accuracy : 0.9249765132197021
Precision: 0.42987152034261245
Recall   : 0.40678824721377915
F1       : 0.418011452368558
ROC-AUC  : 0.8593256076751997


### Tuned Random Forest Results

| Metric | Score |
|---|---:|
| **Accuracy** | **0.924977** |
| **Precision** | **0.429872** |
| **Recall** | **0.406788** |
| **F1 Score** | **0.418011** |
| **ROC-AUC** | **0.859326** |

## 3. XGBoost

In [22]:
xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

xgb_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8],
    "colsample_bytree": [0.8],
    "scale_pos_weight": [3, 5, 8, 12, 14]  # Handles class imbalance
}

xgb_grid = GridSearchCV(
    estimator=xgb,
    param_grid=xgb_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train, y_train)

print("Best F1:", xgb_grid.best_score_)
print("Best Parameters:", xgb_grid.best_params_)

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best F1: 0.4427345597095792
Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'scale_pos_weight': 3, 'subsample': 0.8}


In [24]:
best_xgb = xgb_grid.best_estimator_

y_pred_xgb = best_xgb.predict(X_test)
y_prob_xgb = best_xgb.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall   :", recall_score(y_test, y_pred_xgb))
print("F1       :", f1_score(y_test, y_pred_xgb))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_xgb))

Accuracy : 0.9232317809689975
Precision: 0.422963689892051
Recall   : 0.43667679837892603
F1       : 0.42971086739780656
ROC-AUC  : 0.8621544414433995


### Tuned XGBoost Results

| Metric | XGBoost |
|---|---:|
| **Accuracy** | **0.923232** |
| **Precision** | **0.422964** |
| **Recall** | **0.436677** |
| **F1 Score** | **0.429711** |
| **ROC-AUC** | **0.862154** |

## LightGBM

In [26]:
lgbm = LGBMClassifier(
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgbm_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, -1],
    "learning_rate": [0.05, 0.1],
    "num_leaves": [15, 31],
    "class_weight": [None, {0: 1, 1: 2}, {0: 1, 1: 3}]
}

lgbm_grid = GridSearchCV(
    estimator=lgbm,
    param_grid=lgbm_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

lgbm_grid.fit(X_train, y_train)

print("Best F1:", lgbm_grid.best_score_)
print("Best Parameters:", lgbm_grid.best_params_)

Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best F1: 0.44211257833513296
Best Parameters: {'class_weight': {0: 1, 1: 3}, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'num_leaves': 15}


In [28]:
best_lgbm = lgbm_grid.best_estimator_

y_pred_lgbm = best_lgbm.predict(X_test)
y_prob_lgbm = best_lgbm.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred_lgbm))
print("Precision:", precision_score(y_test, y_pred_lgbm))
print("Recall   :", recall_score(y_test, y_pred_lgbm))
print("F1       :", f1_score(y_test, y_pred_lgbm))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_lgbm))

Accuracy : 0.923298886055563
Precision: 0.4230009871668312
Recall   : 0.4341438703140831
F1       : 0.4285
ROC-AUC  : 0.8621197376894963


### LightGBM Results

| Metric | LightGBM |
|---|---:|
| **Accuracy** | **0.923299** |
| **Precision** | **0.423001** |
| **Recall** | **0.434144** |
| **F1 Score** | **0.428500** |
| **ROC-AUC** | **0.862120** |

## Extra Trees

In [30]:
et = ExtraTreesClassifier(
    random_state=42,
    n_jobs=-1
)

et_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [7, 10, None],
    "min_samples_leaf": [1, 5, 20],
    "class_weight": [None, {0: 1, 1: 2}, {0: 1, 1: 3}]
}

et_grid = GridSearchCV(
    estimator=et,
    param_grid=et_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

et_grid.fit(X_train, y_train)

print("Best F1:", et_grid.best_score_)
print("Best Parameters:", et_grid.best_params_)

Fitting 5 folds for each of 54 candidates, totalling 270 fits


C:\Users\Rupali Waghmare\miniconda3\envs\AI_ML\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best F1: 0.41856543448494216
Best Parameters: {'class_weight': {0: 1, 1: 3}, 'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 100}


In [32]:
best_et = et_grid.best_estimator_

y_pred_et = best_et.predict(X_test)
y_prob_et = best_et.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred_et))
print("Precision:", precision_score(y_test, y_pred_et))
print("Recall   :", recall_score(y_test, y_pred_et))
print("F1       :", f1_score(y_test, y_pred_et))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_et))

Accuracy : 0.9277949268554556
Precision: 0.44612590799031476
Recall   : 0.37335359675785207
F1       : 0.4065085493656922
ROC-AUC  : 0.8475628826923924


### Extra Trees Results

| Metric | Extra Trees |
|---|---:|
| **Accuracy** | **0.927795** |
| **Precision** | **0.446126** |
| **Recall** | **0.373354** |
| **F1 Score** | **0.406509** |
| **ROC-AUC** | **0.847563** |

In [33]:
import joblib

best_xgb = xgb_grid.best_estimator_

joblib.dump(
    best_xgb,
    "../models/best_xgboost.joblib"
)

['../models/best_xgboost.joblib']

In [34]:
joblib.dump(
    encoder,
    "../models/agegroup_encoder.joblib"
)

['../models/agegroup_encoder.joblib']